# 01 - Extracción de datos
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

El primer paso del proyecto es descargar el dataset y entender qué contiene.

**Fuente:** https://sede.vilagarcia.gal/dcsv/C0G6N1Z26P99PL6H

## Paso 0 — Instalar dependencias
Ejecuta esta celda solo la primera vez. Después reinicia el kernel (Kernel → Restart).

In [1]:
%pip install --prefer-binary -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


c:\Users\sagudo.INDRA\Documents\Indra\Fraude Fiscal\.venv\Scripts\python.exe: No module named pip


## Paso 1 — Descargar el fichero CSV

Descargamos el dataset desde la sede electrónica y lo guardamos en `data/raw/`.  
Si ya existe no lo volvemos a descargar (idempotencia básica).

In [ ]:
import requests
from pathlib import Path

URL     = "https://sede.vilagarcia.gal/dcsv/C0G6N1Z26P99PL6H"
DESTINO = Path("../data/raw/contratos_menores.xlsx")

# Comprobamos si el fichero ya existe para no descargarlo de nuevo (idempotencia)
if DESTINO.exists():
    print(f"El fichero ya existe: {DESTINO}")
else:
    print("Descargando...")
    # Petición HTTP GET a la sede electrónica del ayuntamiento
    respuesta = requests.get(URL, timeout=30)
    # Si el servidor devuelve un error (4xx, 5xx) lanza una excepción aquí y paramos
    respuesta.raise_for_status()
    # Escribimos el contenido binario del fichero descargado en disco
    DESTINO.write_bytes(respuesta.content)
    print(f"Guardado en: {DESTINO}  ({len(respuesta.content):,} bytes)")

## Paso 2 — Cargar el CSV con pandas y ver su estructura

Antes de tocar nada, entendemos qué hay dentro.

In [ ]:
import pandas as pd

# Cargamos el Excel en un DataFrame de pandas para inspeccionarlo
df = pd.read_excel(DESTINO)

# Dimensiones del dataset
print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

# Listamos los nombres de columna tal como vienen en el fichero original
print(f"\nNombres de columnas:")
for col in df.columns:
    print(f"  - {col}")

# Contamos nulos por columna — ayuda a detectar campos problemáticos antes de limpiar
print(f"\nValores vacíos:")
print(df.isnull().sum())

print(f"\nPrimeras filas:")
print(df.head())

In [ ]:
# Bloque exploratorio: intentamos leer el fichero como CSV por si tuviera ese formato.
# El fichero resultó ser un Excel (.xlsx), así que este bloque no produce resultados.
# Se conserva como evidencia de la exploración inicial del formato.
for encoding in ["latin-1", "utf-8-sig", "utf-8"]:
    for sep in [",", ";", "\t"]:
        try:
            # Probamos cada combinación de separador y codificación de caracteres
            df = pd.read_csv(DESTINO, sep=sep, encoding=encoding)
            # Si solo hay 1 columna, el separador es incorrecto — seguimos probando
            if df.shape[1] > 1:
                print(f"✓ Funciona con  sep={sep!r}  encoding={encoding!r}")
                print(f"  Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
                print(f"\nNombres de columnas:")
                for col in df.columns:
                    print(f"  - {col}")
                break
        except Exception:
            continue
    else:
        continue
    break

## Paso 3 — Primera revisión de calidad

Cuántos valores vacíos hay en cada columna y qué tipo de dato tiene cada una.

In [ ]:
# Construimos una tabla resumen con el tipo de dato, nulos y porcentaje de nulos de cada columna
resumen = pd.DataFrame({
    "tipo":     df.dtypes,                           # tipo inferido por pandas (str, float64...)
    "nulos":    df.isnull().sum(),                   # cantidad de valores vacíos
    "% nulos":  (df.isnull().mean() * 100).round(1) # porcentaje sobre el total de filas
})
resumen